# Trap

Waits for the 0.00024 BTC algo to quote near the spread, then places a resting ask
just above their bid. When mid ticks up before their next 2s cancel-reprice, their
new bid (`mid − $0.25`) crosses our ask and fills us as makers.

**Pattern recap (from log analysis):**
- Bot runs on ~2s cycle, 100% cancel rate
- Near-spread mode: bids at exactly `mid − $0.25` (confirmed across multiple events)
- Deep mode: bids at `mid − $35` or `mid − $70` (ignore these — too far to trap)
- 93% bid-only, sole order at level 97% of the time
- A static resting ask at best ask anchors mid → bot consistently quotes $0.25 below mid
  (e.g. our ask=62079 → mid=62078.75 → bot bids 62078.5)

**Trap logic:** detect near-spread bid → place ask at `their_bid + TRAP_OFFSET`
(default 0.3, so we sit $0.05 above mid) → any upward mid tick of $0.05+
before their next cancel causes their recomputed `mid − 0.25` to cross our ask.
Log shows mid moves > +$0.5 in 45% of 2-second cycles.

In [ ]:
import sys, os, asyncio, json, math, time
from pathlib import Path
from collections import deque

ROOT_DIR = Path(os.getcwd()).parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import aiohttp, websockets
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display, clear_output
from dotenv import load_dotenv

from lib.execution import ExecutionClient
from lib.coinbase_feed import CoinbaseBookA

load_dotenv(ROOT_DIR / 'keys' / '.env')

TM_REST_URL = os.getenv('BASE_REST_URL', 'https://api.truemarkets.co')
TM_KEY_FILE = str(ROOT_DIR / 'keys' / 'truemarkets-api-key-edd1691b.json')
CB_WS_URL   = os.getenv('COINBASE_WS_URL', 'wss://ws-feed.exchange.coinbase.com')
CB_PRODUCT  = 'BTC-USD'
BASE_ASSET  = 'BTC'
QUOTE_ASSET = 'USDC'

In [ ]:
class TrueMarketsBook:
    _WS_URL = 'wss://api.truex.co/api/v1'
    _HEADERS = {
        'Origin': 'https://truemarkets.co',
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36',
        'Accept-Language': 'en-US,en;q=0.9',
    }

    def __init__(self, symbol='BTC-PYUSD'):
        self.symbol = symbol
        self._bids  = {}
        self._asks  = {}
        self._ready = False

    @property
    def best_bid(self): return max(self._bids) if self._bids else None
    @property
    def best_ask(self): return min(self._asks) if self._asks else None
    @property
    def mid(self):
        b, a = self.best_bid, self.best_ask
        return (b + a) / 2 if b and a else None
    @property
    def is_ready(self): return self._ready

    def _handle_snapshot(self, data):
        self._bids  = {float(b['price']): float(b['qty']) for b in data.get('bids', []) if float(b['qty']) > 0}
        self._asks  = {float(a['price']): float(a['qty']) for a in data.get('asks', []) if float(a['qty']) > 0}
        self._ready = True

    def _handle_update(self, data):
        for b in data.get('bids', []):
            p, q = float(b['price']), float(b['qty'])
            if q == 0: self._bids.pop(p, None)
            else:      self._bids[p] = q
        for a in data.get('asks', []):
            p, q = float(a['price']), float(a['qty'])
            if q == 0: self._asks.pop(p, None)
            else:      self._asks[p] = q

    def find_target_bid(self, size_str, tol=1e-9):
        """Return highest bid level where qty exactly matches size_str, or None."""
        target = float(size_str)
        candidates = [p for p, q in self._bids.items() if abs(q - target) < tol]
        return max(candidates) if candidates else None

    async def run(self):
        backoff = 1
        while True:
            try:
                async with websockets.connect(self._WS_URL, additional_headers=self._HEADERS) as ws:
                    backoff = 1; self._ready = False
                    await ws.send(json.dumps({
                        'type': 'SUBSCRIBE_NO_AUTH', 'item_names': [self.symbol],
                        'channels': ['DEPTH'], 'timestamp': str(int(time.time())),
                    }))
                    async for raw in ws:
                        try: msg = json.loads(raw)
                        except Exception: continue
                        t, d = msg.get('update'), msg.get('data', {})
                        if   t == 'SNAPSHOT': self._handle_snapshot(d)
                        elif t == 'UPDATE':   self._handle_update(d)
            except websockets.ConnectionClosed: pass
            except Exception as e: print(f'Book error: {e}')
            self._ready = False
            await asyncio.sleep(backoff)
            backoff = min(backoff * 2, 30)

### Parameters

In [ ]:
TARGET_SIZE_STR   = '0.00024'    # the algo we're trapping
TRAP_OFFSET       = 0.3          # $ above their bid where we rest our ask
                                  # their formula: bid = mid - 0.25
                                  # so we sit at their_bid + 0.3 = mid + 0.05
                                  # any upward mid tick of $0.05+ fills us
NEAR_MID_MAX_DIST = 15.0         # only trap when their bid is within $15 of mid
                                  # (ignore their deep mid-$35 / mid-$70 backup quotes)
QUOTE_SIZE_BTC    = 0.0004
TICK_SIZE         = 0.1
PRICE_DECIMALS    = 1
LOOP_SECS         = 1.0          # tight loop — their cycle is ~2s, we need to be faster
TRAP_TIMEOUT_SECS = 30.0         # give up on a trap after this long
MAX_POSITION_BTC  = 0.002
DAILY_LOSS_LIMIT  = 10.0
TM_BOOK_SYMBOL    = 'BTC-PYUSD'

### Helpers

In [ ]:
size_str = f'{QUOTE_SIZE_BTC:.8f}'.rstrip('0').rstrip('.')

def fmt_px(p): return f'{p:.1f}'
def ceil_tick(p): return round(math.ceil(p / TICK_SIZE) * TICK_SIZE, PRICE_DECIMALS)

async def cancel_quietly(bot, session, oid):
    if not oid: return
    try: await bot.cancel_order(session, oid)
    except Exception: pass

async def get_full_balances(bot, session):
    data = await bot._get(session, '/v1/conductor/balances')
    total_btc = avail_btc = total_usd = avail_usd = 0.0
    if not data: return None, None, None, None
    for b in data.get('data', []):
        sym   = b.get('symbol', '')
        avail = float(b.get('available', 0) or 0)
        held  = float(b.get('held',      0) or 0)
        if sym == BASE_ASSET:
            total_btc += avail + held; avail_btc += avail
        elif sym in (QUOTE_ASSET, 'PYUSD'):
            total_usd += avail + held; avail_usd += avail
    return total_btc, total_usd, avail_btc, avail_usd

# ── chart series ─────────────────────────────────────────────────────────────
time_series    = []
pnl_series     = []
mid_series     = []
target_bid_series = []   # where the target is quoting
trap_ask_series   = []   # where our trap ask is
fill_times        = []   # elapsed times when fills happen
fill_prices       = []

def update_plot():
    if not time_series: return
    clear_output(wait=True)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

    ax1.plot(time_series, mid_series, color='gray', linewidth=1, alpha=0.7, label='Mid')
    ax1.scatter(time_series, target_bid_series, color='lime', s=10, label='Target bid', zorder=3)
    ax1.scatter(time_series, trap_ask_series,   color='red',  s=10, label='Our trap ask', zorder=3)
    if fill_times:
        ax1.scatter(fill_times, fill_prices, color='gold', s=80, marker='*',
                    label=f'Fill ({len(fill_times)}x)', zorder=5)
    ax1.set_ylabel('Price ($)'); ax1.set_title('Trap — Target bid vs Our ask')
    ax1.legend(loc='upper left', fontsize=8); ax1.grid(True, alpha=0.3)

    ax2.plot(time_series, pnl_series, color='green')
    ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax2.axhline(-DAILY_LOSS_LIMIT, color='red', linestyle='--', linewidth=1)
    ax2.fill_between(time_series, pnl_series, 0,
                     where=[p>=0 for p in pnl_series], alpha=0.15, color='green')
    ax2.fill_between(time_series, pnl_series, 0,
                     where=[p<0  for p in pnl_series], alpha=0.15, color='red')
    ax2.set_ylabel('PnL ($)'); ax2.set_title('Session PnL')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    display(fig); plt.close(fig)

### Trap Loop

In [ ]:
async def run_trap():
    for s in [time_series, pnl_series, mid_series,
              target_bid_series, trap_ask_series,
              fill_times, fill_prices]:
        s.clear()

    bot     = ExecutionClient(key_file=TM_KEY_FILE, base_url=TM_REST_URL)
    book_a  = CoinbaseBookA(ws_url=CB_WS_URL, product_id=CB_PRODUCT)
    tm_book = TrueMarketsBook(symbol=TM_BOOK_SYMBOL)

    feed_cb = asyncio.create_task(book_a.run())
    feed_tm = asyncio.create_task(tm_book.run())

    start_time = time.time()

    # ── trap state ────────────────────────────────────────────────────────────
    trap_oid       = None     # our resting ask order ID
    trap_px        = None     # price we placed it at
    trap_placed_at = None     # when we placed it
    trap_collateral_btc = 0.0 # BTC locked in trap order

    # ── PnL state ─────────────────────────────────────────────────────────────
    initial_btc = initial_usd = None
    prev_total_btc = prev_total_usd = None
    inventory_btc = cash_usdc = maker_volume = 0.0
    fills_count = 0

    try:
        async with aiohttp.ClientSession() as session:
            await bot.authenticate(session)
            print('Authenticated. Cancelling existing orders...')
            await bot.cancel_all(session)

            res = await get_full_balances(bot, session)
            if res[0] is None: raise RuntimeError('Balance fetch failed')
            initial_btc, initial_usd, _, _ = res
            prev_total_btc, prev_total_usd = initial_btc, initial_usd
            print(f'Balances: {initial_btc:.6f} BTC  |  ${initial_usd:.2f} USDC')
            print('Waiting for book snapshot...')

            while not tm_book.is_ready:
                await asyncio.sleep(0.2)
            print(f'Book ready: {tm_book.best_bid:.1f} / {tm_book.best_ask:.1f}')
            print('Hunting...\n')

            loop_n = 0
            while True:
                loop_start = time.time()
                now        = time.time()
                loop_n    += 1

                # ── 1. Book state ─────────────────────────────────────────────
                if not tm_book.is_ready:
                    await asyncio.sleep(LOOP_SECS); continue

                best_bid   = tm_book.best_bid
                best_ask   = tm_book.best_ask
                mid        = tm_book.mid
                target_bid = tm_book.find_target_bid(TARGET_SIZE_STR)

                if not mid or not best_bid or not best_ask:
                    await asyncio.sleep(LOOP_SECS); continue

                mark = book_a.mid or mid

                # ── 2. Balances (every 4 loops to save rate limit) ────────────
                if loop_n % 4 == 1 or trap_oid is None:
                    res = await get_full_balances(bot, session)
                    if res[0] is None:
                        await asyncio.sleep(LOOP_SECS); continue
                    total_btc, total_usd, avail_btc, avail_usd = res

                    # fill detection
                    delta_btc = total_btc - prev_total_btc
                    if delta_btc < -1e-8:
                        fill_px = trap_px or mid
                        maker_volume += abs(delta_btc) * fill_px
                        fills_count  += 1
                        elapsed = now - start_time
                        fill_times.append(elapsed)
                        fill_prices.append(fill_px)
                        trap_oid = trap_px = trap_placed_at = None
                        trap_collateral_btc = 0.0
                        print(f'  ★ FILL #{fills_count}  SELL {abs(delta_btc):.6f} BTC @ ${fill_px:.1f}  cumvol ${maker_volume:.2f}')

                    inventory_btc  = total_btc - initial_btc
                    cash_usdc      = total_usd - initial_usd
                    prev_total_btc = total_btc
                    prev_total_usd = total_usd

                # ── 3. Trap management ────────────────────────────────────────
                action = 'watching'

                # Stale trap: target moved or timed out → cancel
                if trap_oid:
                    timed_out   = (now - trap_placed_at) > TRAP_TIMEOUT_SECS
                    # Cancel if: target gone, target moved far, or timed out
                    target_gone = target_bid is None
                    target_moved = (target_bid is not None and
                                    abs(target_bid - (trap_px - TRAP_OFFSET)) > TICK_SIZE * 2)
                    if timed_out or target_gone or target_moved:
                        await cancel_quietly(bot, session, trap_oid)
                        reason = 'timeout' if timed_out else ('target_gone' if target_gone else 'target_moved')
                        trap_oid = trap_px = trap_placed_at = None
                        trap_collateral_btc = 0.0
                        action = f'cancelled trap ({reason})'

                # No trap: check if we should set one
                if not trap_oid:
                    dist_from_mid = (target_bid - mid) if target_bid is not None else None
                    near_mid = (dist_from_mid is not None and
                                abs(dist_from_mid) <= NEAR_MID_MAX_DIST)

                    if target_bid is not None and near_mid:
                        # Compute trap ask: one TRAP_OFFSET above their bid, ceil to tick
                        proposed_ask = ceil_tick(target_bid + TRAP_OFFSET)

                        # Re-read live book right before placing
                        live_bid = tm_book.best_bid

                        if proposed_ask <= live_bid:
                            action = f'skip (ask {proposed_ask:.1f} <= live_bid {live_bid:.1f})'
                        elif inventory_btc - QUOTE_SIZE_BTC < -MAX_POSITION_BTC:
                            action = f'skip (max short {inventory_btc:+.5f})'
                        elif (avail_btc + trap_collateral_btc) < QUOTE_SIZE_BTC:
                            action = 'skip (no BTC)'
                        else:
                            order = await bot.place_order(
                                session, BASE_ASSET, QUOTE_ASSET,
                                'sell', size_str, 'base', 'limit', fmt_px(proposed_ask)
                            )
                            if order and order.get('order_id'):
                                trap_oid            = order['order_id']
                                trap_px             = proposed_ask
                                trap_placed_at      = now
                                trap_collateral_btc = QUOTE_SIZE_BTC
                                action = (f'TRAP SET  SELL {size_str} @ {proposed_ask:.1f}  '
                                          f'[target_bid={target_bid:.1f}  dist_mid={dist_from_mid:+.2f}]')
                            else:
                                action = 'PLACE FAILED'
                    elif target_bid is not None:
                        action = f'target too deep (dist_mid={dist_from_mid:+.1f})'
                    else:
                        action = 'no target visible'

                # ── 4. Chart + status ─────────────────────────────────────────
                total_pnl = cash_usdc + inventory_btc * mark
                elapsed   = now - start_time

                time_series.append(elapsed)
                pnl_series.append(total_pnl)
                mid_series.append(mid)
                target_bid_series.append(target_bid or float('nan'))
                trap_ask_series.append(trap_px or float('nan'))

                if loop_n % 5 == 1:
                    update_plot()

                trap_age = f'{now - trap_placed_at:.0f}s' if trap_placed_at else '—'
                rl = bot._rl_remaining
                print(
                    f"  {time.strftime('%H:%M:%S')}  "
                    f"mid={mid:.1f}  target={'@'+str(round(target_bid,1)) if target_bid else '—':>10}  "
                    f"trap={'@'+str(trap_px) if trap_px else '—':>10}  age={trap_age:>4}  "
                    f"fills={fills_count}  inv={inventory_btc:+.5f}  "
                    f"pnl=${total_pnl:+.3f}  rl={rl}/100"
                )
                print(f'    {action}')

                # ── 5. Kill switch ─────────────────────────────────────────────
                if total_pnl < -DAILY_LOSS_LIMIT:
                    print(f'\nLoss limit (${total_pnl:.3f}). Cancelling.')
                    await cancel_quietly(bot, session, trap_oid)
                    break

                await asyncio.sleep(max(0.0, LOOP_SECS - (time.time() - loop_start)))

    except asyncio.CancelledError:
        pass
    finally:
        feed_cb.cancel(); feed_tm.cancel()
        print('\nStopped.')

In [ ]:
# Interrupt kernel to stop
await run_trap()